# NNx_SpikeSorting.ipynb

**Description:** Spike-sort NeuroNexus silicon-probe recordings with SpikeInterface. Loads raw MCS or Open Ephys data, builds the probe layout(s), removes bad / out-of-cortex channels, preprocesses (bandpass 300-9000 Hz, CMR), runs one or more sorters per probe (Kilosort4, Lupin, Tridesclous2, wave_clus), then postprocesses (sorting analyzer, quality metrics, Phy export), curates units (automated SNR threshold + manual removal), and saves spike trains, condition info and analyzers.

**Kernel / environment:** run with the venv at `C:\SpikeSorting_mainTools\EPA_sorting\Scripts\python.exe` (Python 3.14). The env and sorter tools live in `C:\SpikeSorting_mainTools`, **not** in this repo:
- wave_clus: `C:\SpikeSorting_mainTools\wave_clus` (set in the wave_clus sorting cell)
- Kilosort: `C:\SpikeSorting_mainTools\EPA_sorting\Kilosort-main` (set in the Kilosort sorting cell)

**Inputs:**
- `base_folder` (str) - root output folder, e.g. `E:\Roy\SpikingSorting_Data`
- `animal` (str) - animal / session name used for output subfolders, e.g. `20260616-p7`
- Raw recordings, one of:
  - MCS: `.raw` files read with `se.read_mcsraw`, one per condition
  - Open Ephys: `Record Node 101` folders read with `se.read_openephys(stream_id='0')`, one per condition (e.g. spon, whiskerActive, TTX)
- `numprobes` (int, 1-4), `probe<N>_type` (probeinterface Probe: `probe88` = 8x8 edge, `probe48` = 4x8, `probe116` = 1x16), `region` (list of str, one per probe, same order as the data)
- `outies0`, `outies1` (list of int) - channel ids to remove per probe (bad or outside cortex)

**Outputs** (all under `{base_folder}\{animal}\{sorter}_sorting\`):
- sorter output folders (one per probe/region)
- `Phy_output` folders (Phy export per probe)
- `SpikeTrains\SpikeTrains_{region}_All_Spikes.csv` - spike times of curated units, in samples (columns = unit ids)
- `ConditionInfo` - condition names with start/end frames of each concatenated recording segment
- `{region}_analyzer` (binary folder), `{region}_curatedanalyzer.zarr`, `{region}_Report`

**Dependencies:** none upstream in this repo; works on raw data. Downstream MATLAB in `SpikeSorting/`: `SpikeTrains_{region}_All_Spikes.csv` is read by `spikessortedinmatlab.m` and `Spike_phase_coherence_spikesorted*.m`; Phy output (`cluster_group.tsv`, `cluster_info.tsv`) and `ConditionInfo` are read by `SpikeSorted_Analysis.m` and `SpikeSorting_Followup/*.m`.

**Inferred contracts (not yet verified):**
- Spike-train CSV times are in samples at the recording rate (30 kHz), not seconds.
- Probe order in `probe<N>_type`, `region` and `outies<N>` must match the channel order in the raw data.
- Some later cells use `regions` (hard-coded `['V1','S1']`) instead of `region`; the two must agree.


In [15]:
base_folder =r'E:\Roy\SpikingSorting_Data'   #where should all the output files be kept? (preferably not assembly, and might be good idea to keep all in same place as raw data)
animal = '20260616-p7'         #what is the name of the animal in your filing system?

IMPORT ALL THE TOOLS YOU'LL NEED

In [14]:

import matplotlib.pyplot as plt
import numpy as np
from pprint import pprint


import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
import spikeinterface.comparison as sc
import spikeinterface.exporters as sexp
import spikeinterface.curation as scur
import spikeinterface.widgets as sw


from probeinterface import Probe, ProbeGroup, get_probe
from probeinterface import generate_linear_probe, generate_multi_shank
from probeinterface import combine_probes
from probeinterface.plotting import plot_probe
from probeinterface.plotting import plot_probegroup


import os
n_cpus = os.cpu_count()
print('CPUs ',n_cpus)

n_jobs = n_cpus - 4


global_job_kwargs = dict(n_jobs=n_jobs, chunk_duration="1s",progress_bar=True)
si.set_global_job_kwargs(**global_job_kwargs)

CPUs  24


LOAD THE DATA
1.) Choose one of the two code cells below, according to the system you recorded with (MCS or OpenEphys)
2.) Add a variable 'recording#' for each file you will include. (A file likely corresponds to a recorded condition. e.g. whisker-900mV or spon)
    Copy and paste the path for each file into the first agrument, so that it reads file_path=r'' or folder_path=r'', and the path within the quotations.
    Most often you will not need to change/add any other arguments. 
3.) Put all the recording variables in the recording list
4.) MATCHING THE ORDER OF THE RECORDING LIST, provide a short name in a notation style of your choosing that annotates the recording. Easiest is to do condition (e.g. 'W900' or 'S'),
        but the choice is yours. Whatever works. ALSO, MAKE SURE THEY ARE ALL UNIQUE.
5.) Run the cell

*Note that if you changed the standard sampling rate of the recording, you'll have to change the variable called fs to whatever that rate is. 


OPTION 1: MCS DATA
- The read funcion requires binary .raw files with the header option, which must be converted from standard mcd using the MC Data Tool.
- DONT GIVE IT MERGED DATA. I DONT KNOW HOW TO HANDLE IT. PLEASE.
- As long as you did not include the digital channels (trigger data), you need no extra arguments. If you did, then you have to include the stream_id argument. 


In [ ]:
#OPTION 1: MCS DATA

#Add all the recordings 
recording0 = se.read_mcsraw(file_path=r'E:\Roy\test_sorting_mcd\20221021-P34-01-multi-900mV.raw')
recording1 = se.read_mcsraw(file_path=r'E:\Roy\test_sorting_mcd\20221021-P34-02-multi-500mV.raw')
recording2 = se.read_mcsraw(file_path=r'E:\Roy\test_sorting_mcd\20221021-P34-03-multi-400mV.raw')

#Make list of the recording variables
recording_list = [recording0, recording1, recording2]#, recording3, recording4, recording5, recording6, recording7, recording8, recording9] #

#Annotate each recording with something simple and clear (order must match that of recording_list, or else you will die unhappy and alone)
rec_conditions = ['W900', 'W500', 'W400']  #'L22_24', 'L22_40','L22_72', 'LW22_1','LW22_24', 'LW22_40','LW22_72']  #

#sampling rate of recording
fs = 20000



############################################
#Nothing to do below this line. Run dat shi#
############################################






#merge any conditions that are multiple segments
print(f"Old Rec List: {recording_list}")
ind = 0
for sub_rec in recording_list:
    segs = sub_rec.get_num_segments()
    
    if segs > 1:
        sub_rec_segs = [ [0]*segs for i in range(segs)]
        for i in range(segs):
            sub_rec_segs[i]= sub_rec.select_segments(i)
            
        
        sub_rec_conc = si.concatenate_recordings(recording_list=sub_rec_segs)
    
        recording_list[ind] = sub_rec_conc
    ind = ind+1
print(f"New Rec List: {recording_list}")
        

#put it together
recording = si.concatenate_recordings(recording_list=recording_list)

#define conidtions
recording.annotate(conditions=rec_conditions)
recording.get_annotation('conditions')


seg_end = []
seg_end_f = []
lastend = 0
lastend_f = 0
for i in range(len(recording_list)):
    item = recording_list[i]
    itemend = item.get_total_duration()+lastend
    itemend_f = item.get_num_frames()+lastend_f
    seg_end.append(itemend)
    seg_end_f.append(itemend_f)
    lastend = itemend
    lastend_f = itemend_f

recording.annotate(segment_endtimes=seg_end)
recording.annotate(segment_endframes=seg_end_f)
recording = si.unsigned_to_signed(recording)
recording.get_annotation('segment_endtimes')

OPTION 2: OPEN EPHYS DATA
        -Include the full file ending in 'Record Node ###'. Probably node 101, or whichever is the full unfiltered recording.
        -Don't change the stream_id = 0 unless you changed the standard recording signal chain, in which case you're on your own chief. 

In [16]:
#OPTION 2: OPEN EPHYS DATA

#Add all the recordings 
recording0 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260616-p7\20260616-p7-spon\Record Node 101', stream_id = '0')
recording1 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260616-p7\20260616-p7-whiskerActive\Record Node 101', stream_id = '0')
recording2 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260616-p7\20260616-p7-whiskerActive_2\Record Node 101', stream_id = '0')
recording3 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260616-p7\20260616-p7-sponTTX\Record Node 101', stream_id = '0')
recording4 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260616-p7\20260616-p7-whiskerActive_TTX\Record Node 101', stream_id = '0')
recording5 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260616-p7\20260616-p7-whiskerActive_TTX_2\Record Node 101', stream_id = '0')


#recording0 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-spon\Record Node 101', stream_id = '0')
#recording1 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-whisker\Record Node 101', stream_id = '0')
#recording2 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40\Record Node 101', stream_id = '0')
#recording3 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40-n-whisker\Record Node 101', stream_id = '0')
#recording4 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-whisker_2\Record Node 101', stream_id = '0')
#recording5 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40_2\Record Node 101', stream_id = '0')
#recording6 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40-n-whisker_2\Record Node 101', stream_id = '0')
#recording7 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-spon_TTX\Record Node 101', stream_id = '0')
#recording8 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-whisker_TTX\Record Node 101', stream_id = '0')
#recording9 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40_TTX\Record Node 101', stream_id = '0')
#recording10 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40-n-whisker_TTX\Record Node 101', stream_id = '0')
#recording11 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-whisker_TTX_2\Record Node 101', stream_id = '0')
#recording11 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40_TTX_2\Record Node 101', stream_id = '0')
#recording12 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260508-p9\20260508-p9-light22_40-n-whisker_TTX_2\Record Node 101', stream_id = '0')



#recording2 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_32\Record Node 101', stream_id = '0')
#recording3 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_32-n-whisker\Record Node 101', stream_id = '0')
#recording5 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_40-n-whisker\Record Node 101', stream_id = '0')
#recording6 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_48\Record Node 101', stream_id = '0')
#recording7 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_48-n-whisker\Record Node 101', stream_id = '0')
#recording8 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_56\Record Node 101', stream_id = '0')
#recording9 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_56-n-whisker\Record Node 101', stream_id = '0')
#recording10 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-whisker_2\Record Node 101', stream_id = '0')
#recording11 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_32_2\Record Node 101', stream_id = '0')
#recording12 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_32-n-whisker_2\Record Node 101', stream_id = '0')
#recording13 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_40_2\Record Node 101', stream_id = '0')
#recording14 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_40-n-whisker_2\Record Node 101', stream_id = '0')
#recording15 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_48_2\Record Node 101', stream_id = '0')
#recording16 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_48-n-whisker_2\Record Node 101', stream_id = '0')
#recording17 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_56_2\Record Node 101', stream_id = '0')
#recording18 = se.read_openephys(folder_path=r'E:\Roy\Silicon Probe Raw Data\20260423-p12\20260423-p12-light22_56-n-whisker_2\Record Node 101', stream_id = '0')



#Make list of the recording variables
recording_list = [recording0,recording1, recording2, recording3, recording4, recording5]#, recording6, recording7, recording8, recording9, recording10]#, recording11, recording12]#, recording13]#,
                 # recording14, recording15, recording16, recording17, recording18] 

#Annotate each recording with something simple and clear (order must match that of recording_list, or else you will die unhappy and alone)
#rec_conditions = ['S', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10',]
#rec_conditions = ['S','W','L22_40','LW22_40','W_2','L22_40_2','LW22_40_2','S_TTX','W_TTX','L22_40_TTX','LW22_40_TTX','L22_40_TTX_2','LW22_40_TTX_2']#,'L22_40','LW22_40','L22_48','LW22_48','L22_56','LW22_56',
                  #'W_2','L22_32_2','LW22_32_2','L22_40_2','LW22_40_2','L22_48_2','LW22_48_2','L22_56_2','LW22_56_2']  
rec_conditions = ['S', 'W', 'W_2', 'S_TTX', 'W_TTX', 'W_TTX_2']


#sampling rate of recording
fs = 30000



############################################
#Nothing to do below this line. Run dat shi#
############################################



#merge any conditions that are multiple segments
print(f"Old Rec List: {recording_list}")
ind = 0
for sub_rec in recording_list:
    segs = sub_rec.get_num_segments()
    
    if segs > 1:
        sub_rec_segs = [ [0]*segs for i in range(segs)]
        for i in range(segs):
            sub_rec_segs[i]= sub_rec.select_segments(i)
            
        
        sub_rec_conc = si.concatenate_recordings(recording_list=sub_rec_segs)
    
        recording_list[ind] = sub_rec_conc
    ind = ind+1
print(f"New Rec List: {recording_list}")
        

#put it together
recording = si.concatenate_recordings(recording_list=recording_list)

#define conidtions
recording.annotate(conditions=rec_conditions)
recording.get_annotation('conditions')


seg_end = []
seg_end_f = []
lastend = 0
lastend_f = 0
for i in range(len(recording_list)):
    item = recording_list[i]
    itemend = item.get_total_duration()+lastend
    itemend_f = item.get_num_frames()+lastend_f
    seg_end.append(itemend)
    seg_end_f.append(itemend_f)
    lastend = itemend
    lastend_f = itemend_f

recording.annotate(segment_endtimes=seg_end)
recording.annotate(segment_endframes=seg_end_f)
recording.get_annotation('segment_endtimes')




Old Rec List: [OpenEphysBinaryRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 36,490,752 samples 
                                   1,216.36s (20.27 minutes) - int16 dtype - 8.70 GiB, OpenEphysBinaryRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 36,072,192 samples 
                                   1,202.41s (20.04 minutes) - int16 dtype - 8.60 GiB, OpenEphysBinaryRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 41,865,216 samples 
                                   1,395.51s (23.26 minutes) - int16 dtype - 9.98 GiB, OpenEphysBinaryRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 19,197,184 samples 
                                   639.91s (10.67 minutes) - int16 dtype - 4.58 GiB, OpenEphysBinaryRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 40,287,744 samples 
                                   1,342.92s (22.38 minutes) - int16 dtype - 9.61 GiB, OpenEphysBinaryRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 33,263,872 sam

[np.float64(1216.3583999999998),
 np.float64(2418.7647999999995),
 np.float64(3814.271999999999),
 np.float64(4454.178133333333),
 np.float64(5797.102933333334),
 np.float64(6905.898666666668)]

CONSTRUCT THE PROBE LAYOUT OF THE EXPERIMENT
    1.) Run the first cell. That one makes the different types of probes that you can choose from
    2.) Fill in the second cell to set the number of probes and type of probes present in the recording. 
        Also make sure to give a region name to each probe (i.e. where was the probe located?). Note that these names must be different. 
        And then run that cell, of course.
    3.) Next choose either the MCS or OpenEphys cell to construct the probes and set all the proper labels. There shouldn't be anything to change in them, so just hit run.
        If you get an error, come to me (Roy), and tell me I'm an idiot and need to fix it (former and latter).
    

In [17]:
probe88 = generate_multi_shank(num_shank=8, num_columns=1, num_contact_per_column=8, xpitch=200, ypitch=100,
                                     contact_shapes='square', contact_shape_params={'width': 17})
probe48 = generate_multi_shank(num_shank=4, num_columns=1, num_contact_per_column=8, xpitch=200, ypitch=100,
                                     contact_shapes='circle', contact_shape_params={"radius":7.5})
probe116 = generate_multi_shank(num_shank=1, num_columns=1, num_contact_per_column=16, xpitch=200, ypitch=100,
                                     contact_shapes='circle', contact_shape_params={"radius":7.5})

In [18]:
#How many probes are in the recording data?
numprobes = 2 #RANGE 1 to 4

#what type of probe is each one?  (probe88 = 8x8 edge, probe48 = 4x8, probe116 = 1x16)
#NOTE: THE ORDER USED HERE MUST MATCH THE DATA
probe0_type = probe88.copy()
probe1_type = probe88.copy()

#What brain region is each probe in?
region = ['V1', 'S1'] #should be a list of strings in the same order as the probes, with no repeated region names


In [ ]:
### MCS RECORDING PROBE SETUP ###

##############################
### Set up the first probe ###
##############################

#assign the proper probe info to probe0
probe0 = probe0_type


#generate probe contact ids (should match your mental layout of the probe) and assign device channel indices (layout should match the data)
if probe1_type.get_contact_count == 16:
   dvidx0 = [11, 6, 5, 12, 1, 16, 3, 14, 9, 8, 7, 10, 13, 4, 15, 2] #device indices according to channel reordering chart
   dvidx0 = list(np.array(dvidx0)-1)
   ids0 = list(np.arange(16,0,-1))
   
elif probe1_type.get_contact_count == 32:
   dvidx0 =[11, 6, 10, 14, 12, 8, 9, 13, 15, 1, 4, 3, 16, 5, 2, 7, 32, 24, 30, 29, 28, 27, 26, 31, 25, 17, 22, 19, 23, 18, 20, 21]  #device indices according to channel reordering chart
   dvidx0 = list(np.array(dvidx0)-1)
   nums0 = np.arange(8,0,-1)
   ids0 = np.array([])
   for i in range(4):
    bumpup = 8*(i+1)
    bumped0 = (nums0)+(bumpup-8)
    ids0 = [*ids0,*bumped0]
   
else:
   dvidx0 = [38, 40, 36, 42, 37, 44, 39, 46, 41, 48, 43, 2, 45, 4, 47, 6, 1, 8, 3, 10, 5, 14, 7, 16, 9, 11, 15, 13, 12, 35, 34, 33, 63, 62, 21, 20, 18, 22, 24, 17, 26, 19, 28, 23, 30, 25, 32, 27, 50, 29, 52, 31, 54, 49, 56, 51, 58, 53, 60, 55, 61, 57, 59]  #device indices according to channel reordering chart
   dvidx0 = list(np.array(dvidx0)-1)
   nums0 = np.arange(8,0,-1)
   ids0 = np.array([])
   
   #make a new probe real quick because the missing reference channel fucks everything up
   shank1 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank2 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank3 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank4 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank5 = generate_linear_probe(num_elec=7, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank6 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank7 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank8 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
   shank2.move([200,0])
   shank3.move([400,0])
   shank4.move([600,0])
   shank5.move([800,100])
   shank5.set_shank_ids([4, 4, 4, 4, 4, 4, 4])
   shank6.move([1000,0])
   shank7.move([1200,0])
   shank8.move([1400,0])
   probe0 = combine_probes([shank1,shank2, shank3, shank4, shank5, shank6, shank7, shank8])
   

   for i in range(8):
    bumpup = 8*(i+1)
    bumped0 = (nums0)+(bumpup-8)
    ids0 = [*ids0,*bumped0]
   ids0.pop(32)

#assign contact and device channel ids
probe0.set_contact_ids(ids0)
probe0.set_device_channel_indices(dvidx0)





##############################
### Set up the second probe ###
##############################



if numprobes == 2:
   #assign the proper probe info to probe1
   probe1  = probe1_type
   probe1.move([200*(probe0.get_shank_count()-1),0])

   #generate probe contact ids (should match your mental layout of the probe) and assign device channel indices (layout should match the data)
   if probe2_type.get_contact_count == 16:
      dvidx1 = list(np.array([11, 6, 5, 12, 1, 16, 3, 14, 9, 8, 7, 10, 13, 4, 15, 2])+probe0.get_contact_count()-1)
      ids1 = list(np.arange(16+probe0.get_contact_count(),probe0.get_contact_count(),-1))
      

   elif probe2_type.get_contact_count == 32:
      dvidx1 =list(np.array([11, 6, 10, 14, 12, 8, 9, 13, 15, 1, 4, 3, 16, 5, 2, 7, 32, 24, 30, 29, 28, 27, 26, 31, 25, 17, 22, 19, 23, 18, 20, 21])+probe0.get_contact_count()-1)
      nums1 = np.arange(8+probe0.get_contact_count(),probe0.get_contact_count(),-1)
      ids1 = np.array([])
      for i in range(4):
         bumpup = 8*(i+1)
         bumped1 = (nums1)+(bumpup-8)
         ids1 = [*ids1,*bumped1]
    

   else:
      
      dvidx1 = list(np.array([38, 40, 36, 42, 37, 44, 39, 46, 41, 48, 43, 2, 45, 4, 47, 6, 1, 8, 3, 10, 5, 14, 7, 16, 9, 11, 15, 13, 12, 35, 34, 33, 63, 62, 21, 20, 18, 22, 24, 17, 26, 19, 28, 23, 30, 25, 32, 27, 50, 29, 52, 31, 54, 49, 56, 51, 58, 53, 60, 55, 61, 57, 59])+probe0.get_contact_count()-1)
      nums1 = np.arange(8+probe0.get_contact_count(),probe0.get_contact_count(),-1)
      ids1 = np.array([])
      
      #make a new probe because the missing reference channel fucks things up
      shank1 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank2 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank3 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank4 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank5 = generate_linear_probe(num_elec=7, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank6 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank7 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)
      shank8 = generate_linear_probe(num_elec=8, ypitch=100, contact_shapes='square', contact_shape_params = {'width': 17},)

      shank2.move([200,0])
      shank3.move([400,0])
      shank4.move([600,0])
      shank5.move([800,100])
      shank5.set_shank_ids([4, 4, 4, 4, 4, 4, 4])
      shank6.move([1000,0])
      shank7.move([1200,0])
      shank8.move([1400,0])
      probe1 = combine_probes([shank1,shank2, shank3, shank4, shank5, shank6, shank7, shank8])
      probe1.move([200*(probe0.get_shank_count()-1),0])

      for i in range(8):
         bumpup = 8*(i+1)
         bumped1 = (nums1)+(bumpup-8)
         ids1 = [*ids1,*bumped1]
      ids1.pop(32) #remove the reference channel index
      
   #assigne contact ids and device indices to probe1   
   probe1.set_contact_ids(ids1)
   probe1.set_device_channel_indices(dvidx1)






#now set the location properties and split the recording by probes
brain_area_property_values = [region[0]]*probe0.get_contact_count() 
probe_property_values = [0]*probe0.get_contact_count() 

if numprobes == 2:
    brain_area_property_values = brain_area_property_values + [region[1]]*probe1.get_contact_count()
    probe_property_values = probe_property_values + [1]*probe1.get_contact_count()

recording.set_property(key='brain_area', values=brain_area_property_values)
recording.set_property(key='probehome', values=probe_property_values)

recording_split = recording.split_by('probehome')
recording_split
recording_split[0] = recording_split[0].set_probe(probe0)
if numprobes == 2:
    recording_split[1] = recording_split[1].set_probe(probe1)

In [19]:
### OPENEPHYS RECORDING PROBE SETUP ###

##############################
### Set up the first probe ###
##############################

#assign the proper probe info to probe0
probe0 = probe0_type

#generate probe contact ids (should match your mental layout of the probe) and device channel ids (layout should match the data)
nums0 = np.arange(8,0,-1)
devnum0 = np.arange(7,-1,-1)
ids0 = np.array([])
dvidx0 = np.array([])

if probe1_type.get_contact_count == 32:
    for i in range(4):
        bumpup = 8*(i+1)
        bumped0 = (nums0)+(bumpup-8)
        bumpidx0 = (devnum0)+(bumpup-8)
        ids0 = [*ids0,*bumped0]
        dvidx0 = [*dvidx0,*bumpidx0]
    probe0.set_contact_ids(ids0)
    probe0.set_device_channel_indices(dvidx0)
else:
    for i in range(8):
        bumpup = 8*(i+1)
        bumped0 = (nums0)+(bumpup-8)
        bumpidx0 = (devnum0)+(bumpup-8)
        ids0 = [*ids0,*bumped0]
        dvidx0 = [*dvidx0,*bumpidx0]

#assign contact ids and device indices to probe0
probe0.set_contact_ids(ids0)
probe0.set_device_channel_indices(dvidx0)
 

###############################
### Set up the second probe ###
###############################

if numprobes == 2:
    #assign the proper probe info
    probe1 = probe1_type
    probe1.move([200*(probe0.get_shank_count()-1),0])

    #generate probe contact ids (should match your mental layout of the probe) and device channel indices (layout should match the data)
    nums1 = np.arange(8+probe0.get_contact_count(),probe0.get_contact_count(),-1)
    devnum1 = np.arange(7,-1,-1)
    ids1 = np.array([])
    dvidx1 = np.array([])

    if probe1_type.get_contact_count == 32:
        for i in range(4):
            bumpup = 8*(i+1)
            bumped1 = (nums1)+(bumpup-8)
            bumpidx1 = (devnum1)+(bumpup-8)
            ids1 = [*ids1,*bumped1]
            dvidx1 = [*dvidx1,*bumpidx1]
    else:
        for i in range(8):
            bumpup = 8*(i+1)
            bumped1 = (nums1)+(bumpup-8)
            bumpidx1 = (devnum1)+(bumpup-8)
            ids1 = [*ids1,*bumped1]
            dvidx1 = [*dvidx1,*bumpidx1]

    
    #assign contact ids and device indices to probe1
    probe1.set_contact_ids(ids1)
    probe1.set_device_channel_indices(dvidx1)



#now set the location properties and split the recording by probes
brain_area_property_values = [region[0]]*probe0.get_contact_count() 
probe_property_values = [0]*probe0.get_contact_count() 

if numprobes == 2:
    brain_area_property_values = brain_area_property_values + [region[1]]*probe1.get_contact_count()
    probe_property_values = probe_property_values + [1]*probe1.get_contact_count()

recording.set_property(key='brain_area', values=brain_area_property_values)
recording.set_property(key='probehome', values=probe_property_values)

recording_split = recording.split_by('probehome')
recording_split
recording_split[0] = recording_split[0].set_probe(probe0)
if numprobes == 2:
    recording_split[1] = recording_split[1].set_probe(probe1)


Below, you can view either probe and it's labels, if you'd like to double check or just enjoy the view. 

BAD CHANNEL REMOVAL:
- Below, please list all channels to be exluded from the sorting (Good idea to remove the ones outside the brain). 
- You will need to list them according to the contact_id number (I recommend plotting the probe with the cell above to make sure you list the correct ones)
- If you only have one probe, you don't need to (please don't) comment out or delete "outties2," the cell won't do anything with it anyway

In [ ]:

#list of channels to remove in the first probe
#outies0 = [1, 9, 17, 18, 25, 26, 33, 34, 41, 42, 49, 50, 51, 57, 58, 59]
#outies0 = [1, 2, 9, 10, 17, 18, 25, 26, 27, 33, 34, 35, 41, 42, 43, 49, 50, 51, 57, 58, 59] 
#outies0 = [1, 2, 3, 4, 5, 9, 10, 11, 12, 17, 18, 19, 25, 26, 27, 33, 34, 35, 41, 42, 43, 44, 49, 50, 51, 52, 57, 58, 59, 60]
#outies0 = [1, 2, 9, 10, 17, 18, 25, 26, 33, 34, 41, 49, 57]
outies0 = []

#list of channel sto remove in the second probe, if it exists
#outies1 = [65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 81, 82, 83, 89, 90, 91, 97, 98, 105, 106, 113, 114, 121, 122]
#outies1 =  [65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 81, 82, 89, 90, 97, 98, 105, 106, 110, 113, 114, 115, 121, 122, 123]
#outies1 = [65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 81, 82, 89, 90, 97, 98, 105, 113, 121]
#outies1 = [65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 81, 82, 89, 90, 97, 98, 105, 106, 113, 114, 121, 122]
outies1 = [65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 81, 82, 83, 84, 89, 90, 91, 97, 98, 105, 106, 113, 121]






### nothing to edit down here ###




prefix = 'CH'

ch_remove0 = [ f'{prefix}{num}' for num in outies0] 


newchans0 = [ f'{prefix}{num}' for num in np.arange(1,probe0.get_contact_count()+1)] 
if numprobes == 2:
    ch_remove1 = [ f'{prefix}{num}' for num in outies1] 
    newchans1 = [ f'{prefix}{num}' for num in np.arange(1+probe0.get_contact_count(),1+probe0.get_contact_count()+probe1.get_contact_count())] 






#first rename the channels because somehow they get all out of order as soon as you add a probe 
recording_split_newchans = {}
recording_split_newchans[0] = recording_split[0].rename_channels(newchans0)

recording_split_gchans = {}
recording_split_gchans[0] = recording_split_newchans[0].remove_channels(ch_remove0)

print(recording_split_gchans[0])
if numprobes == 2:
    recording_split_newchans[1] = recording_split[1].rename_channels(newchans1)
    recording_split_gchans[1] = recording_split_newchans[1].remove_channels(ch_remove1)


Preprocessing Starts Here

In [ ]:
#recording_f = si.bandpass_filter(recording_split_gchans, freq_min=300, freq_max=9000)
#recording_cmr = si.common_reference(recording_f, reference='global', operator='median')
#recording_preprocessed = recording_cmr
recording_ready = recording_split_gchans

In [ ]:
%matplotlib widget
probenum = 0 ##which probe to plot?
w = si.plot_traces({"filtered": recording_f[probenum], "common": recording_cmr[probenum]}, mode='map',segment_index=0, time_range =[1, 2], backend="ipywidgets")

CHANGE THIS SECTION TO REMOVE CHANNELS OUTSIDE CORTEX (PROVIDE MANUALLY IDENTIFIED LIST)

In [ ]:
#get only subsection of recording for testing sorting things

#fs = recording_cmr.get_sampling_frequency()
#recording_sub = recording_good_channels.frame_slice(start_frame=0*fs, end_frame=200*fs)
#recording_sub

In [ ]:
import os
n_cpus = os.cpu_count()
print('CPUs ',n_cpus)

n_jobs = n_cpus - 4

job_kwargs = dict(n_jobs=n_jobs, chunk_duration="2s", progress_bar=True)

In [ ]:
#to compress the data when you save it
#import numcodecs
#compressor = numcodecs.Blosc(cname="zstd", clevel=9, shuffle=numcodecs.Blosc.BITSHUFFLE)
#recording_saved = recording_sub.save(format="zarr", folder=base_folder / "preprocessed_compressed.zarr",
#                                         compressor=compressor,
#                                         **job_kwargs)
#recording_ready[0]

Check For and Choose Sorters

In [ ]:
si.installed_sorters()

In [ ]:
si.available_sorters()

In [ ]:
si.get_sorter_params_description('kilosort4')


Run Sorting (Locally or with Docker/Singlularity)

In [ ]:
sorting_TD2 = {}
sorter = 'TD2'

for group, sub_recording in recording_preprocessed.items():
    sortTD2 = ss.run_sorter('tridesclous2', 
                            sub_recording, 
                            folder=f'{base_folder}\{animal}\TD2_sorting\{region[group]}_baseSorting',
                            verbose=True,
                            apply_preprocessing = False,
                            detection={'detect_threshold':8},
                            apply_merge = False,
                            #singularity_image=True, 
                            job_kwargs=global_job_kwargs
                            )
    sorting_TD2[group] = sortTD2
    
                            

In [ ]:
regions = ['V1','S1']
sorting_WC = {}
si.WaveClusSorter.set_waveclus_path(r'C:\SpikeSorting_mainTools\wave_clus')

for group, sub_recording in recording_ready.items():
    sortWC = ss.run_sorter('waveclus', 
                            sub_recording, 
                            folder=f'{base_folder}\{animal}\WC_sorting\{regions[group]}_baseSorting',
                            verbose=True,
                            #detection={'detect_threshold':8},
                            #singularity_image=False, 
                            #job_kwargs=global_job_kwargs
                            )
    sorting_WC[group] = sortWC                        

In [ ]:
si.KilosortSorter.set_kilosort_path(r'C:\SpikeSorting_mainTools\EPA_sorting\Kilosort-main')

sort_KS4_ThU9L8_250Hz_nt81_mcd10_nc1_batch10_ccgp2 = ss.run_sorter('kilosort4', recording_ready, 
                            folder=f'{base_folder}\\{animal}\\KS4_sorting_ThU9L8_250Hz_nt81_mcd10_nc1_batch10_ccgp2',
                            verbose=True,
                            nblocks = 0,
                            Th_universal = 9,
                            Th_learned = 8,
                            #Th_single_ch = 6,
                            highpass_cutoff = 250,
                            batch_size = 30000*10,
                            #n_pcs = 8,
                            #max_peels = 120,
                            #dmin = 200, 
                            #dminx = 200,
                            nearest_chans = 1,
                            #nearest_templates = 3,
                            ccg_threshold = .2,
                            #templates_from_data = False,
                            #min_template_size = 20,
                            nt = 81,
                            max_channel_distance = 10,
                            #duplicate_spike_ms = .5,
                            whitening_range = 3
                            #docker_image=True, 
                           #job_kwargs=job_kwargs
                            )

In [ ]:
si.get_default_sorter_params('kilosort4')

In [ ]:
si.get_sorter_params_description('lupin')

In [ ]:
sort_Lup_allrads15_Th10_mj1 = ss.run_sorter('lupin', recording_ready, 
                            folder=f'{base_folder}\{animal}\Lup_sorting_allrads15_Th10_mj1',
                            verbose=True,
                            apply_motion_correction = False,
                            detection_radius_um = 15,
                            features_radius_um = 15,
                            split_radius_um = 15,
                            template_radius_um = 15,
                            whitening_radius_um = 15,
                            detect_threshold = 10,
                            template_max_jitter_ms = .1,
                            #template_matching_engine = 'wobble',
                            #docker_image=True, 
                            job_kwargs=job_kwargs
                        )
    #sorting_KS4[group] = sort_KS4

In [ ]:
for group, sub_recording in recording_preprocessed.items():
    sort_Lup = ss.run_sorter('tridesclous2', 
                            sub_recording, 
                            folder=f'{base_folder}\{animal}\sort_Lup_def\{group]}',
                            verbose=True,
                            job_kwargs=job_kwargs
                        )
    sort_Lup_default[group] = sort_Lup

Extracting Waveforms 

POSTPROCESSING

In [ ]:
#sorting = sort_Lup_default
#sorter = "Lup"

In [ ]:
analyzer = {}
metrics = si.get_quality_metric_list()

for group, sub_sorting in sorting.items():
    analyzer[group] = si.create_sorting_analyzer(sorting=sub_sorting, recording=recording_ready[group], format="memory")

    analyzer[group].compute({
    
        'random_spikes': {'max_spikes_per_unit': 1_000},
        'waveforms': {'ms_before': 1.5, 'ms_after':3.5},
        'templates': {'operators':['average','std']},
        'noise_levels': {},
        'spike_amplitudes': {},
        'template_similarity': {},
        'correlograms': {'bin_ms': 0.5},
        'unit_locations': {},
        'principle_components':{}
         })
    
    print(analyzer[group])


In [ ]:
for prb in range(0,numprobes):
    sorting_prb = si.read_sorter_folder(fr'{base_folder}\\{animal}\\KS4_sorting_ThU9L8_250Hz_nt81_mcd10_nc1_batch10_ccgp2\\{prb}')
    sortfolder = fr'{base_folder}\\{animal}\\KS4_sorting_ThU9L8_250Hz_nt81_mcd10_nc1_batch10_ccgp2\\{prb}'
    analyzer = {}
    #metrics = si.get_quality_metric_list()

    analyzer = si.create_sorting_analyzer(sorting=sorting_prb, recording=recording_ready[prb], format="memory")

    analyzer.compute({
        'random_spikes': {'max_spikes_per_unit': 1_000},
        'waveforms': {'ms_before': 1.5, 'ms_after':3.5},
        'templates': {'operators':['average','std']},
        'noise_levels': {},
        'spike_amplitudes': {},
        'correlograms': {'bin_ms': 0.5},
        'unit_locations': {},
        'principal_components':{},
        'template_similarity': {}
         })
    
    print(analyzer)

        
    phyoutput_folder = f'{sortfolder}\\Phy_output' 
    sexp.export_to_phy(analyzer,phyoutput_folder)


    import pandas as pd



    conditions = recording.get_annotation('conditions')
    conditions_endtimes = recording.get_annotation('segment_endframes')

    conditions_starttimes = [x+1 for x in conditions_endtimes[0:len(rec_conditions)-1]]
    conditions_starttimes = [0]+conditions_starttimes

    recording_meta = pd.DataFrame(zip(conditions, conditions_starttimes, conditions_endtimes))
    recording_meta.to_csv(phyoutput_folder+'\ConditionInfo', header=["Conditions", "Condition_StartFrames", "Condition_EndFrames"])


In [ ]:
import pandas as pd



conditions = recording.get_annotation('conditions')
conditions_endtimes = recording.get_annotation('segment_endframes')

conditions_starttimes = [x+1 for x in conditions_endtimes[0:len(rec_conditions)-1]]
conditions_starttimes = [0]+conditions_starttimes

recording_meta = pd.DataFrame(zip(conditions, conditions_starttimes, conditions_endtimes))

for prb in range(0,numprobes):
    phyoutput_folder = fr'E:\Roy\SpikingSorting_Data\20260226-p12\sort_Lup_def\phy{prb}'
    recording_meta.to_csv(phyoutput_folder+'\ConditionInfo', header=["Conditions", "Condition_StartFrames", "Condition_EndFrames"])

In [ ]:
phyoutput_folder = 'E:\\Roy\\SpikingSorting_Data\\20260226-p12\\KS4_sorting_Th108_d150_t64_ts20\\1\\Phy_output'

Start here for 'Automated' curation

In [ ]:
#first curation steps
sorting_clean = {}
for group, sub_sorting in sorting.items():
    cleaning = scur.remove_duplicated_spikes(sub_sorting, censored_period_ms=1.0, method="keep_first_iterative")
    cleaning = scur.remove_excess_spikes(cleaning, recording_ready[group])
    cleaning = scur.remove_redundant_units(cleaning, align=False, remove_strategy="max_spikes")
    sorting_clean[group] = cleaning


print(f"Before Cleaning: \n {sorting}")
print(f"After Cleaning: \n {sorting_clean}")


In [ ]:
analyzer = {}
metrics = si.get_quality_metric_list()

for group, sub_sorting in sorting_clean.items():
    analyzer[group] = si.create_sorting_analyzer(sorting=sorting, recording=recording_ready[group], format="memory")

    analyzer[group].compute({
    
        'random_spikes': {'max_spikes_per_unit': 1_000},
        'waveforms': {'ms_before': 1.5, 'ms_after':3.5},
        'templates': {'operators':['average','std']},
        'noise_levels': {},
        'spike_amplitudes': {},
        'template_similarity': {},
        'correlograms': {'bin_ms': 0.5},
        'unit_locations': {},
         })
    
    print(analyzer[group])


    
   

In [ ]:
sorting_metrics = {}

for group, sub_analyzer in analyzer.items():

    sorting_metrics[group] = sqm.compute_quality_metrics(sub_analyzer,metric_names=[
                                                         'num_spikes',
                                                         'firing_rate',
                                                         'snr',
                                                         'rp_violation',
                                                         'amplitude_cutoff',
                                                         'amplitude_median',
                                                         'amplitude_cv',
                                                         'sd_ratio',
                                                         'noise_cutoff']
                                                         )

In [ ]:
snr_thresh = 4
sd_rat_thresh = 1.0
rp_con_thresh = 0.2

rule = f"snr > {snr_thresh}"# & rp_contamination < {rp_con_thresh}"#& sd_ratio < {sd_rat_thresh}"# 
good_metrics = {}
curated_unit_ids = {}
for group, sub_metrics in sorting_metrics.items():
    good_metrics[group] = sub_metrics.query(rule)

    curated_unit_ids[group] = list(good_metrics[group].index)

    print(curated_unit_ids[group])

In [ ]:
curated_sorting = {}
curated_analyzer = {}
for group, sub_units in curated_unit_ids.items():
    curated_sorting[group] = sorting_clean[group].select_units(sub_units)
    curated_analyzer[group] = analyzer[group].select_units(sub_units)

In [ ]:
%matplotlib widget
si.plot_unit_templates(curated_analyzer[1], backend="ipywidgets")


In [ ]:
si.plot_autocorrelograms(curated_analyzer[1], backend='matplotlib')

In [ ]:
%matplotlib widget
si.plot_amplitudes(curated_analyzer[1], backend='ipywidgets', plot_histograms=True)

In [ ]:
#manually list units to remove

units_bad = {}
units_bad[0] = [13,14,20,23,24, 25, 30]
units_bad[1] = [24, 25, 34, 36, 41, 43, 105, 106]

units_merge = {}
units_merge[0] = [[0,0]] #keep the 1,1 always (or at least a repeated unit present in the list), since you can't pass an empty list
units_merge[1] = [[0,0]]



fully_curated_analyzer = {}
fully_curated_mergesorting = {}
fully_curated_sorting = {}

#for group, sub_analyzer in curated_analyzer.items():
#    fully_curated_analyzer[group] = sub_analyzer.remove_units(remove_unit_ids=units_bad[group])
#    fully_curated_analyzer[group] = fully_curated_analyzer[group].merge_units(merge_unit_groups=units_merge[group], censor_ms=0.2, merging_mode='hard', new_id_strategy='take_first')

for group, sub_sorting in curated_sorting.items():
    fully_curated_sorting[group] = sub_sorting.remove_units(remove_unit_ids=units_bad[group])
    fully_curated_sorting[group] = si.merge_units_sorting(sorting=fully_curated_sorting[group], units_to_merge=units_merge[group], delta_time_ms=0.2, new_unit_ids=list(zip(*units_merge[group]))[0])
    #fully_curated_sorting[group] = si.UnitsSelectionSorting(fully_curated_mergesorting[group])


curated_analyzer
fully_curated_analyzer 

curated_sorting
fully_curated_sorting 

In [ ]:
#I want a data
all_spike_trains = {}

for group,sub_sorting in fully_curated_sorting.items():
    
    probe_spikes = []
    
    for id_i in sub_sorting.get_unit_ids():
        
        unit_train = sub_sorting.get_unit_spike_train(unit_id=id_i)
        probe_spikes.append(unit_train)

    all_spike_trains[group] = probe_spikes


In [ ]:
spikedir = os.makedirs(f"{base_folder}\{animal}\{sorter}_sorting\ConditionInfo")

In [ ]:
import pandas as pd

for group,sub_spikes in all_spike_trains.items():
    all_spike_trains_df = pd.DataFrame(sub_spikes).T
    all_spike_trains_df.to_csv(f"{base_folder}\{animal}\{sorter}_sorting\SpikeTrains\SpikeTrains_{regions[group]}_All_Spikes.csv", header=fully_curated_sorting[group].get_unit_ids()) #need to specify a path

In [ ]:
import pandas as pd



conditions = recording.get_annotation('conditions')
conditions_endtimes = recording.get_annotation('segment_endframes')

conditions_starttimes = [x+1 for x in conditions_endtimes[0:len(rec_conditions)-1]]
conditions_starttimes = [0]+conditions_starttimes

recording_meta = pd.DataFrame(zip(conditions, conditions_starttimes, conditions_endtimes))
recording_meta.to_csv(spikedir, header=["Conditions", "Condition_StartFrames", "Condition_EndFrames"])

In [ ]:
#base_folder = r'C:\Users\rdmis\Documents\PhD\V1\spikeinterface_test'
analyzer_saved = {}
analyzer_curated_saved = {}

for group,sub_analyzer in analyzer.items():
    analyzer_saved[group] = sub_analyzer.save_as(folder=f'{base_folder}\{animal}\{sorter}_sorting\{regions[group]}_analyzer', format='binary_folder')
                                                 
for group,sub_analyzer in analyzer.items():                                                 
    analyzer_curated_saved[group] = sub_analyzer.save_as(folder=f'{base_folder}\{animal}\{sorter}_sorting\{regions[group]}_curatedanalyzer.zarr', format="zarr")

In [ ]:
for group, sub_analyzer in analyzer_curated_saved.items():
    si.export_report(sub_analyzer, output_folder=f'{base_folder}\{animal}\{sorter}_sorting\{regions[group]}_Report')